# TriForge on Kaggle: one 12-hour session of a 2B ternary fine-tune

Settings (right-hand panel): **Accelerator: GPU T4 x2**, **Internet: on**. Add-ons → Secrets: `GITHUB_TOKEN`
(a GitHub token that can read Vedant-Sood/TernaryAI; not needed if the repo is public) and optionally `HF_TOKEN`.

Run with **Save Version → Save & Run All (Commit)** so it keeps going with the browser closed.
To continue a run in the next session, add this notebook's previous output as an input
(Add Input → Your Work → this notebook, or create a dataset from the Output tab and add that), then run again.
`scripts/cloud_session.py` finds the newest checkpoint under `/kaggle/input` and resumes from it.
See `docs/CLOUD_GPU.md` in the repo.

In [ ]:
import os, time
os.environ["TRIFORGE_SESSION_START"] = str(time.time())   # the training cap is computed from this

SPECIALIST = "trimath"   # tricode | trimath | trireason
PRESET = "quick"         # quick (~30 min, measures speed) | partial (full run, several sessions)
BRANCH = "claude/project-thread-psmmpb"   # switch to "main" once the TriForge PR is merged
REPO = "github.com/Vedant-Sood/TernaryAI.git"
WORK = "/kaggle/working/TernaryAI"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
import subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        token = secrets.get_secret("GITHUB_TOKEN")
    except Exception:
        print("no GITHUB_TOKEN secret: cloning without a token (works only if the repo is public)")
    try:
        os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    except Exception:
        pass
except ImportError:
    pass
url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"
if os.path.isdir(WORK):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only", url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORK], check=True)
# never leave the token in .git/config: /kaggle/working is saved as the notebook's output
subprocess.run(["git", "-C", WORK, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
del token, url
%cd {WORK}
!git log --oneline -1

In [ ]:
# Kaggle's preinstalled CUDA torch is kept; only the other dependencies are installed.
!pip install -q -r requirements/base.txt -r requirements/hf.txt "bitsandbytes>=0.43"
!pip install -q --no-deps -e .
!python scripts/check_env.py

In [ ]:
!python scripts/cloud_session.py --specialist {SPECIALIST} --preset {PRESET} --input-root /kaggle/input --export-when-done

In [ ]:
# what this session leaves in its output (Kaggle keeps up to 20 GB of /kaggle/working)
!du -sh /kaggle/working/TernaryAI/checkpoints/* /kaggle/working/TernaryAI/data/* 2>/dev/null
!tail -n 3 checkpoints/{SPECIALIST}/bitnet2b_{PRESET}_t4/metrics.jsonl